In [ ]:
%pip install numpy astropy astroquery matplotlib requests reproject

In [ ]:
from pathlib import Path
import numpy as np
from astroquery.ipac.irsa import Irsa
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u

# Coordenadas de M101
RA = 210.80227
DEC = 54.34895
coord = SkyCoord(RA, DEC, unit="deg", frame="icrs")

# Buscar imagenes que cubran la posicion
resultados = Irsa.query_sia(
    pos=(coord, 1 * u.arcsec),
    collection=["spherex_dr1", "spherex_qr3"]
)

# Filtrar por detector D2
d2 = resultados[
    resultados["energy_bandpassname"] == "SPHEREx-D2"
]

d2.sort("t_min")

# Crear carpeta y guardar los metadatos
carpeta = Path("03_Catalogos")
carpeta.mkdir(exist_ok=True)

for coleccion in ["spherex_dr1", "spherex_qr3"]:
    tabla = d2[d2["obs_collection"] == coleccion]
    tabla.write(
        carpeta / f"M101_{coleccion}_D2.csv",
        format="ascii.csv",
        overwrite=True
    )
    print(f"{coleccion}: {len(tabla)} imagenes")

print("\nTotal D2:", len(d2))

In [ ]:
import io
import requests
from astropy.io import fits

# Cuantas imagenes descargar, repartidas a lo largo del tiempo
N_IMAGENES = 6

if len(d2) < 2:
    raise ValueError("Se necesitan al menos 2 observaciones D2 para esta posicion.")

# Elegir la observacion mas cercana a cada una de N fechas equiespaciadas
# (entre la primera y la ultima disponible, DR1 + QR3).
t_todas = np.asarray(d2["t_min"], dtype=float)
objetivos = np.linspace(t_todas.min(), t_todas.max(), min(N_IMAGENES, len(d2)))
indices = sorted({int(np.argmin(np.abs(t_todas - t))) for t in objetivos})
ids = [str(d2["obs_id"][i]) for i in indices]

# Para elegirlas a mano, descomenta y usa obs_id de los CSV guardados:
# ids = ["2025W50_1A_0005_3", "2026W23_2A_0336_3", ...]

print(f"Observaciones elegidas ({len(ids)}):", ids)

base = Path("04_Datos_FITS")
base.mkdir(exist_ok=True)

obs = []   # informacion de cada observacion descargada

for obs_id in ids:
    filas = d2[d2["obs_id"] == obs_id]

    if len(filas) != 1:
        print(f"No se encontro una fila unica para {obs_id}")
        continue

    fila = filas[0]
    coleccion = str(fila["obs_collection"])
    subcarpeta = base / ("DR1" if coleccion == "spherex_dr1" else "QR3")
    subcarpeta.mkdir(exist_ok=True)

    # URL del FITS original y peticion de recorte
    url = str(fila["access_url"])
    url_cutout = f"{url}?center={RA},{DEC}&size=0.1"

    archivo = subcarpeta / f"{obs_id}_D2_cutout.fits"

    try:
        respuesta = requests.get(url_cutout, timeout=(20, 120))
        respuesta.raise_for_status()

        # Comprobar que el contenido sea FITS
        with fits.open(io.BytesIO(respuesta.content)) as hdul:
            hdul.verify("exception")
            print(obs_id, "Extensiones:", len(hdul))

        archivo.write_bytes(respuesta.content)
        print("Guardado:", archivo)

        t_obs = Time(float(fila["t_min"]), format="mjd")
        fecha = t_obs.strftime("%Y-%m-%d")
        obs.append({
            "obs_id": obs_id,
            "ruta": archivo,
            "t_obs": t_obs,
            "fecha": fecha,
            "titulo": f"{coleccion.replace('spherex_', '').upper()} - {fecha}",
        })

    except Exception as error:
        print("Error al descargar:", obs_id, error)

obs.sort(key=lambda o: o["t_obs"].mjd)   # orden temporal
print(f"\nDescargadas: {len(obs)} de {len(ids)}")

if len(obs) < 2:
    raise RuntimeError("Se necesitan al menos 2 imagenes descargadas para comparar.")

In [ ]:
from pathlib import Path
from astropy.io import fits

for o in obs:
    archivo = o["ruta"]
    print(f"\nArchivo: {archivo.name}")

    if not archivo.exists():
        print("NO SE ENCONTRO EL ARCHIVO")
        continue

    print(f"Tamano: {archivo.stat().st_size / 1024**2:.2f} MB")

    try:
        with fits.open(archivo) as hdul:
            hdul.verify("exception")
            hdul.info()

            if "IMAGE" in hdul:
                print("Imagen cientifica encontrada")
                print("Dimensiones:", hdul["IMAGE"].data.shape)
            else:
                print("No se encontro la extension IMAGE")

    except Exception as e:
        print("Error al leer el FITS:", e)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import (
    ImageNormalize, PercentileInterval, AsinhStretch
)

def leer_recorte(ruta, coord):
    """Lee un recorte SPHEREx: imagen, WCS espacial y banda espectral en `coord`."""
    with fits.open(ruta) as hdul:
        imagen = hdul["IMAGE"].data.astype(float).copy()
        header = hdul["IMAGE"].header

        # WCS espacial
        wcs = WCS(header).celestial
        x, y = wcs.world_to_pixel(coord)

        # WCS espectral (metodo oficial NASA/IPAC)
        wcs_espectral = WCS(header, fobj=hdul, key="W")
        wcs_espectral.sip = None
        longitud_onda, ancho_banda = wcs_espectral.pixel_to_world(x, y)

    meta = {
        "x": float(x),
        "y": float(y),
        "lambda_um": float(longitud_onda.to_value(u.um)),
        "ancho_um": float(ancho_banda.to_value(u.um)),
    }
    return imagen, wcs, meta

# =============================================
# 1. LEER CADA RECORTE
# =============================================
datos = []
for o in obs:
    imagen, wcs, meta = leer_recorte(o["ruta"], coord)
    datos.append({"imagen": imagen, "wcs": wcs, "meta": meta})

    print(f"\n{o['titulo']}")
    print(f"Longitud de onda: {meta['lambda_um']:.4f} um")
    print(f"Ancho de banda: {meta['ancho_um']:.4f} um")
    print(f"Posicion pixel: ({meta['x']:.2f}, {meta['y']:.2f})")

lams = [d["meta"]["lambda_um"] for d in datos]
print(f"\nRango de longitudes de onda: {min(lams):.3f} - {max(lams):.3f} um")

# =============================================
# 2. COMPARACION INICIAL (cada imagen en su propio WCS)
# =============================================
n = len(obs)
ncols = min(n, 3)
nrows = int(np.ceil(n / ncols))
fig = plt.figure(figsize=(4.5 * ncols, 4.5 * nrows))

for i, (o, d) in enumerate(zip(obs, datos), start=1):
    meta = d["meta"]
    norm = ImageNormalize(
        d["imagen"],
        interval=PercentileInterval(99),
        stretch=AsinhStretch()
    )
    ax = fig.add_subplot(nrows, ncols, i, projection=d["wcs"])
    ax.imshow(d["imagen"], origin="lower", cmap="magma", norm=norm)

    # Marcar la posicion de M101
    ax.scatter(meta["x"], meta["y"], marker="+", color="cyan",
               s=130, linewidths=1.5)

    ax.set_title(f"{o['titulo']}\n{meta['lambda_um']:.3f} um", fontsize=9)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("M101_D2_comparacion_inicial.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
from reproject import reproject_interp

n = len(obs)
wcs_ref = datos[0]["wcs"]
forma = datos[0]["imagen"].shape

# ==========================================
# 1. ALINEAR TODAS LAS IMAGENES A LA PRIMERA
# ==========================================
alineadas = [datos[0]["imagen"]]
comunes = np.isfinite(datos[0]["imagen"])

for d in datos[1:]:
    al, huella = reproject_interp(
        (d["imagen"], d["wcs"]), wcs_ref, shape_out=forma
    )
    alineadas.append(al)
    # Conservar solamente la region comun a TODAS las imagenes
    comunes &= np.isfinite(al) & (huella > 0.99)

if not np.any(comunes):
    raise ValueError("Las imagenes no tienen una region comun valida.")

print("Cobertura comun:", f"{100 * comunes.mean():.1f}%")

# ==========================================
# 2. MISMA ESCALA DE INTENSIDAD
# ==========================================
valores = np.concatenate([a[comunes] for a in alineadas])
vmin, vmax = np.percentile(valores, [1, 99.5])
norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())

# ==========================================
# 3. VISUALIZAR LA COMPARACION
# ==========================================
ncols = min(n, 3)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(
    nrows, ncols, figsize=(4.5 * ncols, 4.5 * nrows),
    subplot_kw={"projection": wcs_ref}, squeeze=False
)
axes = axes.ravel()

for ax, al, o, d in zip(axes, alineadas, obs, datos):
    im = ax.imshow(np.where(comunes, al, np.nan), origin="lower",
                   cmap="magma", norm=norm)
    ax.set_title(f"{o['titulo']}\n{d['meta']['lambda_um']:.3f} um", fontsize=9)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")

for ax in axes[n:]:
    ax.axis("off")

fig.colorbar(im, ax=list(axes[:n]), shrink=0.8,
             label="Intensidad (unidades del FITS)")
plt.savefig("M101_comparacion_alineada.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image

# ==========================================
# 1. PREPARAR LAS IMAGENES
# ==========================================
imagenes = [np.ma.masked_where(~comunes, a) for a in alineadas]

etiquetas = [
    f"M101 | {o['titulo']} | {d['meta']['lambda_um']:.2f} um"
    for o, d in zip(obs, datos)
]

cmap = plt.colormaps["magma"].copy()
cmap.set_bad("black")

# ==========================================
# 2. CREAR LA ANIMACION (misma escala `norm` que la celda anterior)
# ==========================================
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection=wcs_ref)

im = ax.imshow(
    imagenes[0],
    origin="lower",
    cmap=cmap,
    norm=norm,
    interpolation="nearest"
)

ax.set_xlabel("Ascension recta (RA)")
ax.set_ylabel("Declinacion (Dec)")

titulo = ax.set_title(etiquetas[0])

def actualizar(frame):
    im.set_data(imagenes[frame])
    titulo.set_text(etiquetas[frame])
    return im, titulo

FPS = 2
animacion = FuncAnimation(
    fig,
    actualizar,
    frames=len(imagenes),
    interval=int(1000 / FPS),
    repeat=True,
    blit=False
)

# ==========================================
# 3. GUARDAR COMO GIF
# ==========================================
carpeta = Path("05_Analisis")
carpeta.mkdir(exist_ok=True)

salida = carpeta / "M101_blink.gif"

animacion.save(str(salida), writer=PillowWriter(fps=FPS))
plt.close(fig)

print("Animacion guardada en:", salida)

if max(lams) - min(lams) > 0.05:
    print("\nAviso: las imagenes estan en longitudes de onda distintas; "
          "el cambio de brillo puede ser espectral, no solo temporal.")

display(Image(filename=str(salida)))